# 11주차 과제 — Git 작업

- **작업 내용을 확인**하기 위해 만든 ipynb 파일입니다.
- 작업 코드는 `코드_공유.py`, `성능확인.ipynb` 파일에 정리해 주십시오.
- 브랜치별 작업 순서는 PDF 의 도식을 함께 확인하십시오.

**예측 과제** — 2013년 뉴욕 출발 항공편이 **도착 지연 30분 이상**이 될지 분류합니다.

> **주의** — 출발 이후에 알 수 있는 값(`dep_delay`, `dep_time`, `arr_time`, `air_time`)은
> 피처로 사용하지 않습니다. 아래 데이터 세팅 코드가 그렇게 구성되어 있습니다.

---
- **데이터 세팅 코드**
- 아래 코드를 그대로 사용합니다.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV

# 2013년 뉴욕 출발 항공편 표본 50,000건입니다.
# 항공편 정보에 기상 정보와 기체 정보를 미리 결합해 두었습니다.
df = pd.read_csv('flights_ml.csv')

NUM = ['month', 'day', 'sched_hour', 'distance', 'temp', 'humid',
       'wind_speed', 'precip', 'visib', 'seats', 'engines']
CAT = ['carrier', 'origin']

X = pd.get_dummies(df[NUM + CAT], columns=CAT)
X = X.fillna(X.median(numeric_only=True))
y = df['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)

---
- **A_branch 에서 이루어질 작업**
    - A 작업자는 **DecisionTreeClassifier** 를 사용한 모델링을 수행합니다.
    - `GridSearchCV` 로 하이퍼파라미터 탐색을 수행합니다. `cv=5` 를 적용합니다.
    - 탐색과 평가의 기준은 **F1 score** 로 합니다.
    - 최적 조합을 찾은 뒤 **Feature Importance 상위 10개**를 시각화합니다.

- 하이퍼파라미터 후보는 아래와 같습니다.
    - `criterion` : ['gini', 'entropy']
    - `max_depth` : [5, 10]
    - `min_samples_leaf` : [1, 5, 20]

---
- **B_branch 에서 이루어질 작업**
    - B 작업자는 **XGBClassifier** 를 사용한 모델링을 수행합니다.
    - `GridSearchCV` 로 하이퍼파라미터 탐색을 수행합니다. `cv=5` 를 적용합니다.
    - 탐색과 평가의 기준은 **F1 score** 로 합니다.
    - 최적 조합을 찾은 뒤 **Feature Importance 상위 10개**를 시각화합니다.

- 하이퍼파라미터 후보는 아래와 같습니다.
    - `max_depth` : [3, 6, 9]
    - `n_estimators` : [100, 300]
    - `learning_rate` : [0.1, 0.05]

---
- **B_branch 작업 중 Admin_branch 에서 이루어질 작업**
    - Admin 작업자는 A 작업자의 **DecisionTree 하이퍼파라미터 후보를 조정**합니다.
    - `max_depth` 후보를 **[5, 10, 15]** 로 변경하고 `main` 에 반영합니다.

- **B_branch 에서 함께 이루어지는 실험**
    - B 작업자는 자신의 브랜치에서 **DecisionTree 의 `max_depth` 후보를 [3, 7] 로 바꿔** 실험합니다.
    - **Admin 과 같은 줄을 다르게 수정하므로, B 작업자의 Pull Request 에서 충돌이 발생합니다.**
    - 충돌은 **Admin 이 정한 값 [5, 10, 15] 을 채택**하여 해결합니다.

---
- **Pull Request 로 병합합니다**

브랜치를 로컬에서 바로 병합하지 않고, **GitHub 의 Pull Request 를 통해** 병합합니다.

| PR | 방향 | 내용 |
|---|---|---|
| **PR 1** | `A_branch` → `main` | DecisionTree 모델링 코드 |
| **PR 2** | `B_branch` → `main` | XGBoost 모델링 코드 — **충돌 발생 · 직접 해결** |
| **PR 3** | `Tester_branch` → `main` | 성능 비교 노트북 |

- 각 PR 에는 **제목과 설명**을 작성합니다.
- **PR 2 는 충돌을 해결한 뒤 병합**합니다. 충돌 없이 병합된 경우 인정하지 않습니다.
- 세 PR 모두 **병합 완료(Merged)** 상태여야 합니다.

---
- **Tester_branch 에서 이루어질 작업**
    - 병합이 끝난 `코드_공유.py` 의 코드 **전체를** `성능확인.ipynb` 로 옮겨 실행합니다.
    - 두 모델의 **F1 score 와 Accuracy** 를 비교하는 **막대 그래프**를 출력합니다.
    - 마지막 셀에서 **커밋 이력을 출력**합니다 (아래 참고).

```python
import subprocess
print(subprocess.run(['git', 'log', '--oneline', '-n', '20'],
                     capture_output=True, text=True).stdout)
```

> 이 출력의 커밋 해시가 **제출 캡처 화면의 해시와 일치**해야 정답으로 인정됩니다.